In [12]:
# pip install anthropic


In [13]:
from google.colab import userdata
ANTHROPIC_API_KEY=userdata.get('ANTHROPIC_API_KEY')

In [14]:
import anthropic

client = anthropic.Anthropic()

def ask(prompt):
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=512,
        system="You are a helpful support assistant. Be direct and factual.",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

In [18]:
orders_db = {
    "4471": {
        "status": "shipped",
        "carrier": "UPS",
        "eta": "2 days"
    },
    "4472": {
        "status": "processing",
        "carrier": None,
        "eta": None
    },
    "4473": {
        "status": "delivered",
        "carrier": "FedEx",
        "eta": "delivered"
    },
    "4474": {
        "status": "shipped",
        "carrier": "DHL",
        "eta": "4 days"
    },
    "4475": {
        "status": "processing",
        "carrier": None,
        "eta": None
    },
    "4476": {
        "status": "cancelled",
        "carrier": None,
        "eta": None
    },
    "4477": {
        "status": "shipped",
        "carrier": "USPS",
        "eta": "1 day"
    },
    "4478": {
        "status": "delayed",
        "carrier": "UPS",
        "eta": "5 days"
    },
    "4479": {
        "status": "delivered",
        "carrier": "DHL",
        "eta": "delivered"
    },
    "4480": {
        "status": "shipped",
        "carrier": "FedEx",
        "eta": "3 days"
    },
    "4481": {
        "status": "processing",
        "carrier": None,
        "eta": None
    },
    "4482": {
        "status": "returned",
        "carrier": "USPS",
        "eta": None
    },
    "4483": {
        "status": "shipped",
        "carrier": "DHL",
        "eta": "2 days"
    },
    "4484": {
        "status": "delivered",
        "carrier": "UPS",
        "eta": "delivered"
    },
    "4485": {
        "status": "delayed",
        "carrier": "FedEx",
        "eta": "6 days"
    }
}

In [19]:
def get_order_status(order_id):
    return orders_db.get(order_id, {"error": "No order found with that ID"})

get_order_status_schema = {
    "name": "get_order_status",
    "description": (
        "Looks up the current status of a customer order by its ID. "
        "Use this any time a question depends on current order data "
        "rather than general policy information."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string", "description": "The order ID to look up, e.g. '4471'"},
        },
        "required": ["order_id"],
    },
}

In [21]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
tools = [get_order_status_schema]
messages = [{"role": "user", "content": "What is the status of order #4499?"}]

while True:
    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=512,
        tools=tools,
        messages=messages,
    )

    if response.stop_reason != "tool_use":
        break

    messages.append({"role": "assistant", "content": response.content})

    tool_results = []
    for block in response.content:
        if block.type != "tool_use":
            continue
        try:
            result = get_order_status(**block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(result),
            })
        except Exception as e:
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": f"Error: {e}",
                "is_error": True,
            })

    messages.append({"role": "user", "content": tool_results})

print("".join(b.text for b in response.content if b.type == "text"))

/tmp/ipykernel_553/3641487644.py:8: DeprecationWarning: The model 'claude-sonnet-4-5' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client.messages.create(


I'm sorry, but I couldn't find any order with ID #4499 in the system. This could mean:
- The order number might be incorrect
- The order may not exist yet
- There could be a typo in the order number

Could you please double-check the order number and try again?


In [22]:
def run_agent(user_input, tools, tool_map, max_iterations=6):
    messages = [{"role": "user", "content": user_input}]

    for _ in range(max_iterations):
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=512,
            tools=tools,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            return response.content[0].text

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                function = tool_map[block.name]
                output = function(**block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(output),
                })
        messages.append({"role": "user", "content": tool_results})

    return "Stopped after max iterations without a final answer."

In [23]:
class Agent:
    def __init__(self, tools, tool_map, max_iterations=6):
        self.tools = tools
        self.tool_map = tool_map
        self.max_iterations = max_iterations
        self.messages = []

    def run(self, user_input):
        self.messages.append({"role": "user", "content": user_input})

        for _ in range(self.max_iterations):
            response = client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=512,
                tools=self.tools,
                messages=self.messages,
            )
            self.messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason != "tool_use":
                return response.content[0].text

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    function = self.tool_map[block.name]
                    output = function(**block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(output),
                    })
            self.messages.append({"role": "user", "content": tool_results})

        return "Stopped after max iterations without a final answer."